In [ ]:
import numpy as np
from scipy.special import jv
import jax
import jax.numpy as jnp
import dynamiqs as dq
from scipy.sparse.linalg import LinearOperator, eigs
dq.set_precision('double')

In [ ]:
# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 3.8 * 2 * np.pi #GHz
w_b = 2*w_a#7.05 * 2 * jnp.pi #GHz

phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2 

phi_b = 0.2
Z_b = 2 * phi_quantum**2 * phi_b**2 

E_J = 37 * 2 * np.pi
kappa_b = 0.1 * 2 * np.pi

w_d = w_b
w_p = 2 * w_a - w_d

epsilon_p = 0.1*np.pi
g2 = jv(1,epsilon_p)*E_J*phi_a**2*phi_b # (Leon's paper)
epsilon_d= 8 *g2

In [ ]:
n_a = 12
n_b = 6

In [ ]:
# ------------------------------------------------------------
# build static operators once
# ------------------------------------------------------------

a = dq.destroy(n_a)
b = dq.destroy(n_b)

Ia = dq.eye(n_a)
Ib = dq.eye(n_b)

a_tot = dq.tensor(a, Ib)
b_tot = dq.tensor(Ia, b)

na_tot = dq.tensor(dq.number(n_a), Ib)
nb_tot = dq.tensor(Ia, dq.number(n_b))

varphi_a_op_tot = dq.tensor(
    phi_a * (a + a.dag()),
    Ib,
)
varphi_b_op_tot = dq.tensor(
    Ia,
    phi_b * (b + b.dag()),
)

H0 = w_a * na_tot + w_b * nb_tot
H_pump = -2*E_J*np.sin(epsilon_p)*(dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)
drive = lambda t: epsilon_d * jnp.cos(w_d * t)
H_drive = dq.modulated(drive, b_tot + b_tot.dag())
H = H0 + H_drive + H_pump

jump_op = np.sqrt(2 * kappa_b) * b_tot
readout_op = a  # for ptrace(rho, 0)

In [ ]:
T = 2*np.pi / w_b
tsave = np.linspace(0.0, T, 2)

def mat_to_vec(rho_mat):
    return rho_mat.reshape(-1, order='F')

N=n_a*n_b
def vec_to_mat(rho_vec, N):
    return rho_vec.reshape((N, N), order='F')


def propagate_one_period(rho_vec):
    rho_mat = dq.asqarray(vec_to_mat(rho_vec, N), dims=(n_a, n_b))
    result = dq.mesolve(H, [jump_op], rho_mat, tsave)
    rho_mat = (result.states[-1]).to_numpy()
    rho_vec = mat_to_vec(rho_mat)
    return rho_vec

id_mat = np.eye(N, dtype=np.complex128)
id_vec = mat_to_vec(id_mat)
def propagate_one_period_projected(rho_vec):
    rho_mat_np = vec_to_mat(rho_vec, N)
    tr = np.trace(rho_mat_np)
    rho_mat_np -= (tr/N) * id_mat
    rho_mat = dq.asqarray(rho_mat_np, dims=(n_a, n_b))
    result = dq.mesolve(H, [jump_op], rho_mat, tsave)
    rho_mat = (result.states[-1]).to_numpy()
    tr = np.trace(rho_mat)
    rho_vec = mat_to_vec(rho_mat)
    return rho_vec - (tr/N) * id_vec

def project_trace_zero(rho_vec):
    rho_mat = vec_to_mat(rho_vec, N)
    tr = np.trace(rho_mat)
    return rho_vec - (tr / N) * id_vec
dim = N*N
UF = LinearOperator(
    shape=(dim, dim),
    matvec = propagate_one_period_projected,
    dtype=np.complex128
)


In [ ]:
rho_mat = (dq.tensor(dq.coherent_dm(n_a, 1), dq.coherent_dm(n_b, 1))).to_jax()
rho_vec = mat_to_vec(rho_mat)
vec_to_mat(propagate_one_period_projected(rho_vec), N)

In [ ]:
v0 = np.random.randn(dim) + 1j*np.random.randn(dim)
v0 = project_trace_zero(v0)

vals, vecs = eigs(
    UF,
    k=4,
    which='LM',
    v0=v0,
    ncv=30,
    tol=1e-9,
    maxiter=1000
)

In [ ]:
dq.mepropagator(H, [jump_op], tsave)